In [2]:
import pandas as pd

In [3]:
dt = pd.read_csv('network_metrics.csv')

In [4]:
dt.info()
dt.head(3)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 180 entries, 0 to 179
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      180 non-null    object 
 1   timestamp        180 non-null    object 
 2   latency_ms       180 non-null    float64
 3   throughput_mbps  180 non-null    float64
 4   packet_loss_pct  179 non-null    float64
 5   congestion_pct   180 non-null    float64
 6   city             180 non-null    object 
 7   profile          180 non-null    object 
dtypes: float64(4), object(4)
memory usage: 11.4+ KB


,customer_id,timestamp,latency_ms,throughput_mbps,packet_loss_pct,congestion_pct,city,profile
0,C001,2026-10-18 08:00:00,140.48,3.91,5.51,91.95,Benghazi,Home-10
1,C001,2026-10-18 09:00:00,99.88,6.87,5.16,93.72,Benghazi,Home-10
2,C001,2026-10-18 10:00:00,134.70,3.98,3.80,96.54,Benghazi,Home-10


In [5]:
dt["timestamp"]=pd.to_datetime(dt["timestamp"],errors="coerce")
dt.dtypes

,0
customer_id,object
timestamp,datetime64[ns]
latency_ms,float64
throughput_mbps,float64
packet_loss_pct,float64
congestion_pct,float64
city,object
profile,object


In [6]:
dt[dt.isna().any(axis=1)] #check for Na by row

,customer_id,timestamp,latency_ms,throughput_mbps,packet_loss_pct,congestion_pct,city,profile
175,C059,2026-10-18 09:00:00,55.83,43.73,NaN,47.62,Misrata,Home-50


In [7]:
duplicates = dt[dt.duplicated(keep=False)] #Check for all duplicates instances
duplicatescount = dt.duplicated().sum()

print("Duplicate rows:", duplicatescount)
duplicates

Duplicate rows: 0


,customer_id,timestamp,latency_ms,throughput_mbps,packet_loss_pct,congestion_pct,city,profile


In [8]:
subduplicates = dt[dt.duplicated(subset=["customer_id", "timestamp"],keep=False)]
subdupcount = dt.duplicated(subset=["customer_id", "timestamp"]).sum()

print("Duplicate customers and timestamps:", subdupcount)
subduplicates

Duplicate customers and timestamps: 0


,customer_id,timestamp,latency_ms,throughput_mbps,packet_loss_pct,congestion_pct,city,profile


In [9]:
negative_values = dt[(dt["latency_ms"] < 0) | (dt["throughput_mbps"] < 0)]
invalid_packet_loss = dt[dt["packet_loss_pct"].notna() & ((dt["packet_loss_pct"] < 0) |(dt["packet_loss_pct"] > 100))]
invalid_congestion = dt[ dt["congestion_pct"].notna() & ((dt["congestion_pct"] < 0) |(dt["congestion_pct"] > 100))]

print("Missing values:", int(dt.isna().sum().sum())) #NaN at the row 175 packet loss.
print("Negative latency/throughput values:", len(negative_values))
print("Invalid packet-loss values:", len(invalid_packet_loss))
print("Invalid congestion values:", len(invalid_congestion))

Missing values: 1
Negative latency/throughput values: 0
Invalid packet-loss values: 0
Invalid congestion values: 0


In [10]:
dt = dt.sort_values(by=["customer_id", "timestamp"]).reset_index(drop=True)
dt.head(10)

,customer_id,timestamp,latency_ms,throughput_mbps,packet_loss_pct,congestion_pct,city,profile
0,C001,2026-10-18 08:00:00,140.48,3.91,5.51,91.95,Benghazi,Home-10
1,C001,2026-10-18 09:00:00,99.88,6.87,5.16,93.72,Benghazi,Home-10
2,C001,2026-10-18 10:00:00,134.70,3.98,3.80,96.54,Benghazi,Home-10
3,C002,2026-10-18 08:00:00,75.79,11.58,1.39,61.14,Tripoli,Home-20
4,C002,2026-10-18 09:00:00,79.43,5.51,2.16,72.05,Tripoli,Home-20
5,C002,2026-10-18 10:00:00,72.78,7.84,2.44,79.11,Tripoli,Home-20
6,C003,2026-10-18 08:00:00,69.86,16.39,1.43,56.18,Misrata,Home-50
7,C003,2026-10-18 09:00:00,79.95,26.16,1.99,81.12,Misrata,Home-50
8,C003,2026-10-18 10:00:00,68.85,19.91,1.44,60.12,Misrata,Home-50
9,C004,2026-10-18 08:00:00,73.63,44.03,0.95,73.08,Al Bayda,Business-100


In [11]:
latest_records = (dt.dropna(subset=["customer_id", "timestamp"]).groupby("customer_id", as_index=False).tail(1).reset_index(drop=True)) #Check latest customer records

print("Total measurements:", len(dt))
print("Latest customer records:", len(latest_records))
print("Unique customers:", latest_records["customer_id"].nunique())

Total measurements: 180
Latest customer records: 60
Unique customers: 60


In [13]:
quality_indicators = {
    "total_measurements": len(dt),
    "unique_customers": dt["customer_id"].nunique(),
    "latest_customer_records": len(latest_records),
    "missing_values": int(dt.isna().sum().sum()),
    "missing_packet_loss": int(
        dt["packet_loss_pct"].isna().sum()
    ),
    "invalid_latency_or_throughput": len(negative_values),
    "invalid_packet_loss": len(invalid_packet_loss),
    "invalid_congestion": len(invalid_congestion)
}

quality_indicators

{'total_measurements': 180,
 'unique_customers': 60,
 'latest_customer_records': 60,
 'missing_values': 1,
 'missing_packet_loss': 1,
 'invalid_latency_or_throughput': 0,
 'invalid_packet_loss': 0,
 'invalid_congestion': 0}

In [14]:
dt.to_csv("network_metrics_cleaned.csv",index=False)

In [15]:
check = pd.read_csv("network_metrics_cleaned.csv")

print(check.shape)
print(check.head())

(180, 8)
  customer_id            timestamp  latency_ms  throughput_mbps  \
0        C001  2026-10-18 08:00:00      140.48             3.91   
1        C001  2026-10-18 09:00:00       99.88             6.87   
2        C001  2026-10-18 10:00:00      134.70             3.98   
3        C002  2026-10-18 08:00:00       75.79            11.58   
4        C002  2026-10-18 09:00:00       79.43             5.51   

   packet_loss_pct  congestion_pct      city  profile  
0             5.51           91.95  Benghazi  Home-10  
1             5.16           93.72  Benghazi  Home-10  
2             3.80           96.54  Benghazi  Home-10  
3             1.39           61.14   Tripoli  Home-20  
4             2.16           72.05   Tripoli  Home-20  


In [16]:
latest_records.to_csv("network_metrics_latest_per_customer.csv",index=False)

In [17]:
test = pd.read_csv("network_metrics_latest_per_customer.csv")

print(test.shape)
print(test.head())

(60, 8)
  customer_id            timestamp  latency_ms  throughput_mbps  \
0        C001  2026-10-18 10:00:00      134.70             3.98   
1        C002  2026-10-18 10:00:00       72.78             7.84   
2        C003  2026-10-18 10:00:00       68.85            19.91   
3        C004  2026-10-18 10:00:00       67.01            57.19   
4        C005  2026-10-18 10:00:00       27.30             8.77   

   packet_loss_pct  congestion_pct      city       profile  
0             3.80           96.54  Benghazi       Home-10  
1             2.44           79.11   Tripoli       Home-20  
2             1.44           60.12   Misrata       Home-50  
3             1.27           62.78  Al Bayda  Business-100  
4             0.11           15.50  Benghazi       Home-10  


In [18]:
target_city = "Misrata"
target_timestamp = pd.Timestamp("2026-10-18 09:00:00")

reference_values = dt.loc[
    (dt["city"] == target_city) &
    (dt["timestamp"] == target_timestamp) &
    (dt["packet_loss_pct"].notna()),
    "packet_loss_pct"]

print(reference_values)
print("Number of reference records:", len(reference_values))

7      1.99
19     0.45
31     3.65
43     0.04
55     0.23
67     0.61
79     0.73
91     5.64
103    0.62
115    0.56
127    1.06
139    0.19
151    4.04
163    0.24
Name: packet_loss_pct, dtype: float64
Number of reference records: 14


In [32]:
replacement_value = round(reference_values.mean(),2)

print("Replacement packet-loss value:", (replacement_value))

Replacement packet-loss value: 1.43


In [33]:
dt.loc[175, "packet_loss_pct"] = replacement_value

In [34]:
dt.loc[175]

,175
customer_id,C059
timestamp,2026-10-18 09:00:00
latency_ms,55.83
throughput_mbps,43.73
packet_loss_pct,1.43
congestion_pct,47.62
city,Misrata
profile,Home-50


In [36]:
dt.to_csv("network_metrics_cleanedd.csv",index=False)

In [53]:
check = pd.read_csv("network_metrics_cleanedd.csv")

print(check.shape)
print(check.head(5))
print('\n')
print(dt.loc[175])
print("\n Missing values:", int(dt.isna().sum().sum()))


(180, 8)
  customer_id            timestamp  latency_ms  throughput_mbps  \
0        C001  2026-10-18 08:00:00      140.48             3.91   
1        C001  2026-10-18 09:00:00       99.88             6.87   
2        C001  2026-10-18 10:00:00      134.70             3.98   
3        C002  2026-10-18 08:00:00       75.79            11.58   
4        C002  2026-10-18 09:00:00       79.43             5.51   

   packet_loss_pct  congestion_pct      city  profile  
0             5.51           91.95  Benghazi  Home-10  
1             5.16           93.72  Benghazi  Home-10  
2             3.80           96.54  Benghazi  Home-10  
3             1.39           61.14   Tripoli  Home-20  
4             2.16           72.05   Tripoli  Home-20  


customer_id                       C059
timestamp          2026-10-18 09:00:00
latency_ms                       55.83
throughput_mbps                  43.73
packet_loss_pct                   1.43
congestion_pct                   47.62
city            